In [11]:
from ultralytics import YOLO
import cv2
import numpy as np

In [12]:
model = YOLO('yolov8n.pt')

In [13]:
img_path = '../data/raw/8-2-first.jpg'
img = cv2.imread(img_path)

In [14]:
result = model(img)[0]


0: 480x640 13 cars, 35.0ms
Speed: 2.3ms preprocess, 35.0ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)


In [15]:
cars = []
for box in result.boxes:
    cls = int(box.cls[0])
    if cls == 2: #car
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cars.append((x1, y1, x2, y2))

In [16]:
cars.sort(key= lambda b: b[0])

widths = [x2 - x1 for (x1, y1, x2, y2) in cars]
avg_width = np.mean(widths)

In [17]:
THRESHOLD = 1.2 * avg_width

In [18]:
for (x1, y1, x2, y2) in cars:
    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 255), 2)

In [19]:
for i in range(len(cars) - 1):
        x1_a, y1_a, x2_a, y2_a = cars[i]
        x1_b, y1_b, x2_b, y2_b = cars[i + 1]

        gap = x1_b - x2_a

        if gap > THRESHOLD:
            # Draw the parking lot
            x_start = x2_a
            x_end = x1_b
            y_top = min(y1_a, y1_b)
            y_bottom = max(y2_a, y2_b)

            cv2.rectangle(img, (x_start, y_top), (x_end, y_bottom), (0, 255, 0), 2)
            cv2.putText(img, "Park Here", (x_start + 5, y_top - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

In [20]:
cv2.imwrite("../data/processed/resultado.jpg", img)


True